In [1]:
import os
import joblib
import pandas as pd
from ucimlrepo import fetch_ucirepo
print(os.getcwd())

/home/s2026aj05655/workspace/ml_assignment2


In [2]:
# Load the Dry Bean dataset from UCI (id=602)
beans = fetch_ucirepo(id=602)
X = beans.data.features
y = beans.data.targets['Class']   # 7 bean types

# Quick look at the data
print("Dataset shape:", X.shape)
print("\nClass counts:")
print(y.value_counts())
print("\nTotal missing values:", X.isnull().sum().sum())
X.head()

Dataset shape: (13611, 16)

Class counts:
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64

Total missing values: 0


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

test_df = X_test.copy()
test_df['Class'] = y_test
test_df.to_csv('test_data.csv', index=False)

print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (10888, 16)  Test: (2723, 16)


In [4]:
# Import the five classifiers and helper tools
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

# Scale-sensitive models (LR, KNN, SVM) get a StandardScaler in a pipeline.
# Tree and Naive Bayes don't need scaling.
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'Naive Bayes': GaussianNB(),
    'SVM': make_pipeline(StandardScaler(), CalibratedClassifierCV(SVC(random_state=42), ensemble=False)),
}

# Train each model on the training data
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained")

Logistic Regression trained
Decision Tree trained
KNN trained
Naive Bayes trained
SVM trained


In [5]:
# Evaluation metrics
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

# For each model: predict on the test set and compute the 6 metrics.
# 'macro' averaging treats all 7 classes equally (important for imbalanced data).
# AUC uses one-vs-rest ('ovr') because this is a 7-class problem.
results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro'),
        'Precision': precision_score(y_test, y_pred, average='macro'),
        'Recall': recall_score(y_test, y_pred, average='macro'),
        'F1': f1_score(y_test, y_pred, average='macro'),
        'MCC': matthews_corrcoef(y_test, y_pred),
    })

# Comparison table of all models
results_df = pd.DataFrame(results).round(4)
results_df

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9207,0.9948,0.9349,0.9314,0.9329,0.9042
1,Decision Tree,0.8917,0.9448,0.9084,0.9088,0.9085,0.8691
2,KNN,0.9166,0.9833,0.9320,0.9271,0.9293,0.8992
3,Naive Bayes,0.7639,0.9672,0.7744,0.7694,0.7677,0.7154
4,SVM,0.9221,0.9910,0.9359,0.9330,0.9343,0.9059


In [6]:
os.makedirs('model', exist_ok=True)

for name, model in models.items():
    filename = 'model/' + name.replace(' ', '_') + '.joblib'
    joblib.dump(model, filename)
    print("Saved", filename)

# Also save the metrics table for reference
results_df.to_csv('model/metrics.csv', index=False)

Saved model/Logistic_Regression.joblib
Saved model/Decision_Tree.joblib
Saved model/KNN.joblib
Saved model/Naive_Bayes.joblib
Saved model/SVM.joblib
